In [1]:
from openpyxl import load_workbook
import pandas as pd
import unicodedata
import re

In [2]:
df = pd.read_excel('Libro1.xlsx')
df.columns = df.columns.str.strip()

wb = load_workbook('Libro1.xlsx')
ws = wb['BASE DE DATOS']

fecha_inicio = 'Fecha de Inicio del Contrato de Interventoría'
fecha_fin = 'Fecha de Finalización del Contrato de Interventoría'

headers = [
    cell.value.strip() if isinstance(cell.value, str) else cell.value
    for cell in ws[1]
]

In [3]:
col_inicio = headers.index(fecha_inicio) + 1
col_fin = headers.index(fecha_fin) + 1

def tiene_color(celda):
    fill = celda.fill

    if fill is None:
        return False

    if fill.fill_type is None:
        return False

    color = fill.fgColor.rgb

    if color in [None, '00000000', 'FFFFFFFF']:
        return False

    return True

estados = []

for row in range(2, ws.max_row + 1):
    celda_inicio = ws.cell(row=row, column=col_inicio)
    celda_fin = ws.cell(row=row, column=col_fin)

    if tiene_color(celda_inicio) or tiene_color(celda_fin):
        estados.append('FINALIZO')
    else:
        estados.append('ACTIVO')

df['Contrato Estado'] = estados

In [4]:

df = df.dropna(
    subset=[
        'Nombres y Apellidos',
        'No. Cedula de ciudadanía',
        'No. Contrato de Interventoría'
    ],
    how='all'
)

In [5]:
df['Nombres y Apellidos'] = (
    df['Nombres y Apellidos']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    ) if pd.notna(x) else x
)
df['Nombres y Apellidos'] = df['Nombres y Apellidos'].str.replace(r'[^A-ZÑ ]', '', regex=True)

In [6]:
df['No. Cedula de ciudadanía'] = (
    df['No. Cedula de ciudadanía']
    .astype('string')
    .str.replace(r'\D', '', regex=True)
)
df['No. Cedula de ciudadanía'] = df['No. Cedula de ciudadanía'].replace('', pd.NA)
df['No. Cedula de ciudadanía'] = pd.to_numeric(
    df['No. Cedula de ciudadanía'],
    errors='coerce'
)

In [7]:
col = 'No. Contrato de Interventoría'

df[col] = (
    df[col]
    .astype('string')
    .str.upper()
    .str.replace(r'-20\d{2}$', '', regex=True)
    .str.replace(r'\bNO\.\s*', '', regex=True)
    .str.replace('-', '', regex=False)
    .str.replace(r'\D', '', regex=True)
)

df[col] = df[col].fillna('').replace('', '0')
df.loc[df[col].str.len() < 3, col] = '0'
df.loc[df[col].str.fullmatch(r'0+', na=False), col] = '0'

In [8]:
df['Año del Contrato de Interventoría'] = (
    df['Año del Contrato de Interventoría']
    .astype('string')
    .str.extract(r'(19\d{2}|20\d{2})')[0]
    .fillna('0')
)

In [9]:
FECHA_VACIA = '0000-00-00'

MESES = {
    'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04',
    'MAYO': '05', 'JUNIO': '06', 'JULIO': '07', 'AGOSTO': '08',
    'SEPTIEMBRE': '09', 'SETIEMBRE': '09', 'OCTUBRE': '10',
    'NOVIEMBRE': '11', 'DICIEMBRE': '12',
}

MESES_CORTO = {
    'ENE': '01', 'FEB': '02', 'MAR': '03', 'ABR': '04',
    'MAY': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08',
    'SEP': '09', 'OCT': '10', 'NOV': '11', 'DIC': '12',
}

RE_SOLO_TEXTO = re.compile(r'[A-ZÁÉÍÓÚÑ\s,\.]+')
RE_ANIO_MALO = re.compile(r'^0(\d{3})-(\d{2})-(\d{2})$')
RE_FECHA_ISO = re.compile(r'(\d{4}-\d{2}-\d{2})')
RE_FECHA_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})/\(?(\d{4})')
RE_FECHA_DOBLE_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})//(\d{2})')
RE_TEXTO_CON_FECHA = re.compile(r'(\d{1,2})\s+(?:DE\s+)?([A-ZÁÉÍÓÚÑ]+)\s+(?:DE|DEL)?\s+(\d{4})')
RE_FECHA_CORTA = re.compile(r'(\d{1,2})-([A-ZÁÉÍÓÚÑ]{3})-(\d{4})')
RE_MES_ANIO = re.compile(r'^([A-ZÁÉÍÓÚÑ]+)\s+DE\s+(\d{4})$')


def convertir_fecha_texto(valor):
    if pd.isna(valor):
        return FECHA_VACIA

    valor = re.sub(r'\s+', ' ', str(valor).upper().strip())

    if valor in ('00/00/0000', '-'):
        return FECHA_VACIA

    # Si solo tiene letras, espacios, comas o puntos, no es una fecha
    if RE_SOLO_TEXTO.fullmatch(valor):
        return FECHA_VACIA

    # 0201-01-28 -> 2001-01-28
    m = RE_ANIO_MALO.search(valor)
    if m:
        anio, mes, dia = m.groups()
        return f'2{anio}-{mes}-{dia}'

    # 2016-07-28 00:00:00 -> 2016-07-28
    m = RE_FECHA_ISO.search(valor)
    if m:
        return m.group(1)

    # FECHA PROBABLE 08/12/2021 / REINICIO 08/08/2022 / 09/12/(2019
    m = RE_FECHA_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 17/03//21 -> 2021-03-17
    m = RE_FECHA_DOBLE_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'20{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 07 ENERO DE 2020 / 14 OCTUBRE DEL 2022 / 1 DE DICIEMBRE 2022
    m = RE_TEXTO_CON_FECHA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # ESTIMADO 6-FEB-2023
    m = RE_FECHA_CORTA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES_CORTO.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # FEBRERO DE 2019 -> 2019-02-00
    m = RE_MES_ANIO.search(valor)
    if m:
        mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-00'

    # Todo lo que no se pudo convertir queda como fecha por defecto
    return FECHA_VACIA


df['Fecha de Inicio del Contrato de Interventoría'] = (
    df['Fecha de Inicio del Contrato de Interventoría']
    .apply(convertir_fecha_texto)
)

df['Fecha de Finalización del Contrato de Interventoría'] = (
    df['Fecha de Finalización del Contrato de Interventoría']
    .apply(convertir_fecha_texto)
)

In [10]:
col_contrato = 'No. Contrato de Interventoría'
col_anio = 'Año del Contrato de Interventoría'
col_objeto = 'Objeto del Contrato  de Interventoría'

df[col_objeto] = (
    df[col_objeto]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

def descripcion_mas_completa(grupo):
    descripciones = grupo.dropna().astype('string').str.strip()
    descripciones = descripciones[descripciones != '']

    if descripciones.empty:
        return pd.NA

    return descripciones.loc[descripciones.str.len().idxmax()]

objeto_por_contrato = (
    df
    .groupby([col_contrato, col_anio])[col_objeto]
    .transform(descripcion_mas_completa)
)

df[col_objeto] = objeto_por_contrato

palabras_objeto = df[col_objeto].fillna('').str.findall(r'\b\w+\b').str.len()
df = df[palabras_objeto >= 15].copy()

In [11]:
col_cargo = 'Cargo que desempeña en el Contrato de Interventoría'

df[col_cargo] = (
    df[col_cargo]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df[col_cargo] = df[col_cargo].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    ) if pd.notna(x) else x
)

df[col_cargo] = (
    df[col_cargo]
    .str.replace(r'^[^A-ZÑ0-9]+', '', regex=True)
    .str.replace(r'[^A-ZÑ0-9]+$', '', regex=True)
)

# Homologar abreviaturas, errores comunes y variaciones
df[col_cargo] = (
    df[col_cargo]
    .str.replace(r'\bING\.\b', 'INGENIERO', regex=True)
    .str.replace(r'\bING\b', 'INGENIERO', regex=True)
    .str.replace(r'\bINGENIERA\b', 'INGENIERO', regex=True)

    .str.replace(r'\bESP\.\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESP\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILAISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILSITA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILISAT\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESECIALISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bEPECIALISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECAILISTA\b', 'ESPECIALISTA', regex=True)

    .str.replace(r'\bPROF\.\b', 'PROFESIONAL', regex=True)
    .str.replace(r'\bPROF\b', 'PROFESIONAL', regex=True)
    .str.replace(r'\bPROFSIONAL\b', 'PROFESIONAL', regex=True)
    .str.replace(r'\bPROFESINAL\b', 'PROFESIONAL', regex=True)

    .str.replace(r'\bDIRECTORA\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIREC\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIRE\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIR\b', 'DIRECTOR', regex=True)

    .str.replace(r'\bINTEVENTORIA\b', 'INTERVENTORIA', regex=True)
    .str.replace(r'\bRESIEDENTE\b', 'RESIDENTE', regex=True)
    .str.replace(r'\bAUDITORA\b', 'AUDITOR', regex=True)
    .str.replace(r'\bINSPECTORA\b', 'INSPECTOR', regex=True)
    .str.replace(r'\bABOGADA\b', 'ABOGADO', regex=True)
    .str.replace(r'\bTOPOGRAFA\b', 'TOPOGRAFO', regex=True)

    .str.replace(r'\bCALIDA\b', 'CALIDAD', regex=True)
    .str.replace(r'\bDECALIDAD\b', 'DE CALIDAD', regex=True)
    .str.replace(r'\bIUNTERNO\b', 'INTERNO', regex=True)
    .str.replace(r'\bASREGURAMIENTO\b', 'ASEGURAMIENTO', regex=True)

    .str.replace(r'\bGEOTECNICA\b', 'GEOTECNIA', regex=True)
    .str.replace(r'\bHIDRAULICO\b', 'HIDRAULICA', regex=True)

    .str.replace(r'\bDE LA INTERVENTORIA\b', 'DE INTERVENTORIA', regex=True)
    .str.replace(r'\bDE EL\b', 'DEL', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()

    .str.replace(r'\bDIRCTOR\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIRETOR\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIRECTRO\b', 'DIRECTOR', regex=True)
    .str.replace(r'\bDIIRECTOR\b', 'DIRECTOR', regex=True)

    .str.replace(r'\bINGENIRO\b', 'INGENIERO', regex=True)
    .str.replace(r'\bINGNIERO\b', 'INGENIERO', regex=True)
    .str.replace(r'\bINGENIEOR\b', 'INGENIERO', regex=True)
    .str.replace(r'\bINGENIEROA\b', 'INGENIERO', regex=True)

    .str.replace(r'\bAUXILAR\b', 'AUXILIAR', regex=True)
    .str.replace(r'\bAUXLIAR\b', 'AUXILIAR', regex=True)
    .str.replace(r'\bAUXILIARL\b', 'AUXILIAR', regex=True)

    .str.replace(r'\bRESIDNETE\b', 'RESIDENTE', regex=True)
    .str.replace(r'\bRESIEDENTE\b', 'RESIDENTE', regex=True)
    .str.replace(r'\bRECIDENTE\b', 'RESIDENTE', regex=True)

    .str.replace(r'\bINTERVETORIA\b', 'INTERVENTORIA', regex=True)
    .str.replace(r'\bINTERVENOTRIA\b', 'INTERVENTORIA', regex=True)
    .str.replace(r'\bINTERVETNORIA\b', 'INTERVENTORIA', regex=True)
    .str.replace(r'\bINTEVENTORIA\b', 'INTERVENTORIA', regex=True)

    .str.replace(r'\bESPECIALSITA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPCIALISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILISTA\b', 'ESPECIALISTA', regex=True)
    .str.replace(r'\bESPECILAISTA\b', 'ESPECIALISTA', regex=True)

    .str.replace(r'\bPROFESONAL\b', 'PROFESIONAL', regex=True)
    .str.replace(r'\bPROFESINAL\b', 'PROFESIONAL', regex=True)
    .str.replace(r'\bPROFSIONAL\b', 'PROFESIONAL', regex=True)

    .str.replace(r'\bAMBIETAL\b', 'AMBIENTAL', regex=True)
    .str.replace(r'\bAMBIENTA\b', 'AMBIENTAL', regex=True)

    .str.replace(r'\bGEOTECIA\b', 'GEOTECNIA', regex=True)
    .str.replace(r'\bGEOTECNA\b', 'GEOTECNIA', regex=True)

    .str.replace(r'\bHIDRAULCA\b', 'HIDRAULICA', regex=True)
    .str.replace(r'\bHIDRAUILCA\b', 'HIDRAULICA', regex=True)

    .str.replace(r'\bPAVIMENTO\b', 'PAVIMENTOS', regex=True)
)

# Si el cargo parece ser texto del objeto del contrato, marcarlo como sin cargo
df.loc[
    df[col_cargo].astype('string').str.len().gt(120) |
    df[col_cargo].astype('string').str.startswith('INTERVENTORIA PARA', na=False),
    col_cargo
] = 'SIN CARGO'

# Normalizaciones puntuales
df.loc[
    df[col_cargo].astype('string').str.fullmatch(r'DIRECTOR( DE)? INTERVENTORIA', na=False),
    col_cargo
] = 'DIRECTOR DE INTERVENTORIA'

df.loc[
    df[col_cargo].astype('string').str.fullmatch(r'INGENIERO RESIDENTE( DE)? INTERVENTORIA', na=False),
    col_cargo
] = 'INGENIERO RESIDENTE DE INTERVENTORIA'

df.loc[
    df[col_cargo].astype('string').str.fullmatch(r'INGENIERO AUXILIAR( DE)? INTERVENTORIA', na=False),
    col_cargo
] = 'INGENIERO AUXILIAR DE INTERVENTORIA'

df.loc[
    df[col_cargo].astype('string').str.fullmatch(r'ESPECIALISTA AMBIENTAL|ESPECIALISTA EN AMBIENTAL', na=False),
    col_cargo
] = 'ESPECIALISTA AMBIENTAL'

df.loc[
    df[col_cargo].astype('string').str.fullmatch(r'AUDITOR( INTERNO)?( DE)? CALIDAD', na=False),
    col_cargo
] = 'AUDITOR DE CALIDAD'

# Vacíos o solo números
df.loc[
    df[col_cargo].isna() |
    df[col_cargo].astype('string').str.strip().eq('') |
    df[col_cargo].astype('string').str.fullmatch(r'\d+', na=False),
    col_cargo
] = 'SIN CARGO'


def clave_cargo(valor):
    if pd.isna(valor):
        return 'SIN CARGO'

    valor = str(valor).upper().strip()
    valor = re.sub(r'\s+', ' ', valor)

    conectores = {'DE', 'DEL', 'LA', 'EL', 'LOS', 'LAS', 'Y', 'EN'}

    palabras = [
        palabra
        for palabra in valor.split()
        if palabra not in conectores
    ]

    return ' '.join(palabras)


def cargo_mas_completo(grupo):
    cargos = grupo.dropna().astype('string').str.strip()
    cargos = cargos[cargos != '']

    if cargos.empty:
        return 'SIN CARGO'

    return cargos.loc[cargos.str.len().idxmax()]


clave = df[col_cargo].apply(clave_cargo)

cargo_por_clave = (
    df
    .groupby(clave)[col_cargo]
    .transform(cargo_mas_completo)
)

df[col_cargo] = cargo_por_clave

# Revisión final
df.loc[
    df[col_cargo].isna() |
    df[col_cargo].astype('string').str.strip().eq('') |
    df[col_cargo].astype('string').str.fullmatch(r'\d+', na=False),
    col_cargo
] = 'SIN CARGO'

In [12]:
col_estado = 'Contrato Estado'
col_participacion = 'Participación TOTAL en el proyecto (%)'

df[col_participacion] = (
    df[col_participacion]
    .astype('string')
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df[col_participacion] = pd.to_numeric(
    df[col_participacion],
    errors='coerce'
)

# Si viene como decimal entre 0 y 1: 0.05 significa 5%
df.loc[
    df[col_participacion].between(0, 1, inclusive='both'),
    col_participacion
] = df[col_participacion] * 100

# Todo mayor a 100 queda en 100
df.loc[
    df[col_participacion] > 100,
    col_participacion
] = 100

# Todo menor a 0 o vacío queda en 0
df.loc[
    df[col_participacion].isna() | (df[col_participacion] < 0),
    col_participacion
] = 0

# Si el contrato está finalizado, también asegurar 0 cuando quedó mal
mascara_finalizado = (
    df[col_estado]
    .astype('string')
    .str.upper()
    .str.strip()
    .isin(['FINALIZADO', 'FINALIZO', 'FALSO'])
)

df.loc[
    mascara_finalizado &
    (
        df[col_participacion].isna() |
        ~df[col_participacion].between(0, 100, inclusive='both')
    ),
    col_participacion
] = 0

# Dejar con dos decimales
df[col_participacion] = df[col_participacion].round(2)

In [13]:
col_interventoria = 'INTERVENTORÍA'

df[col_interventoria] = (
    df[col_interventoria]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df.loc[
    df[col_interventoria].isna() |
    df[col_interventoria].astype('string').str.strip().eq('') |

    # Solo números o decimales
    df[col_interventoria].astype('string').str.fullmatch(r'\d+(?:[,.]\d+)?', na=False) |

    # Fechas o textos con fecha:
    # 27/012/2024, 29-0-2023, 8 DE MAYO DEL 2023, 2020-12-31
    df[col_interventoria].astype('string').str.contains(
        r'\d{1,2}[/\-]\d{1,3}[/\-]\d{2,4}|\d{4}-\d{1,2}-\d{1,2}|\d{1,2}\s+(?:DE\s+)?[A-ZÁÉÍÓÚÑ]+\s+(?:DE|DEL)\s+\d{4}',
        regex=True,
        na=False
    ) |

    # Duraciones: 2,5 MESES / 4 MESES A PARTIR...
    df[col_interventoria].astype('string').str.contains(
        r'\b\d+(?:[,.]\d+)?\s*(?:MES|MESES|DIA|DIAS|AÑO|AÑOS)\b',
        regex=True,
        na=False
    ) |

    # NIT o códigos tipo 901575984-8
    df[col_interventoria].astype('string').str.fullmatch(r'\d{6,12}-\d', na=False),
    col_interventoria
] = 'SIN DEFINIR'

In [16]:
col_unidad = 'UNIDAD EJECUTORA'

df[col_unidad] = (
    df[col_unidad]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df[col_unidad] = df[col_unidad].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    ) if pd.notna(x) else x
)

valores_invalidos_unidad = (
    df[col_unidad].isna() |
    df[col_unidad].astype('string').str.strip().eq('') |
    df[col_unidad].astype('string').str.fullmatch(r'\|', na=False) |
    df[col_unidad].astype('string').str.fullmatch(
        r'\d{4}-\d{2}-\d{2}(?: 00:00:00)?',
        na=False
    )
)

df.loc[valores_invalidos_unidad, col_unidad] = 'SIN DEFINIR'

correcciones_unidad = {
    'DIRECCION TERRIORIAL': 'DIRECCION TERRITORIAL',

    'SUBDIRECCCION DE GESTION INTEGRAL DE CARRETERAS NACIONALES':
        'SUBDIRECCION DE GESTION INTEGRAL DE CARRETERAS NACIONALES',

    'GESTION INTERGRAL DE CARRETERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTERGAL DE CARRETERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GETSION INTEGRAL DE CARRETERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION NTGRAL DE CARRETERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',

    'GESTION INTEGRAL DE CARRETRAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRTERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRERAS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRETERASS NACIONALES':
        'GESTION INTEGRAL DE CARRETERAS NACIONALES',

    'RED NACIONAL DE CARRTERAS':
        'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETARAS NACIONALES':
        'RED NACIONAL DE CARRETERAS',
    'RED CARRETERAS NACIONALES':
        'RED NACIONAL DE CARRETERAS',
    'RED DE CARRETERAS NACIONALES':
        'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETERAS':
        'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETERAS NACIONALES':
        'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL DE CARRETERAS NACIONALES':
        'RED NACIONAL DE CARRETERAS',

    'GRANDES PRPYECTOS': 'GRANDES PROYECTOS',
    'GRADES PROYECTOS': 'GRANDES PROYECTOS',
    'GRANSDES PROYECTOS': 'GRANDES PROYECTOS',

    'MODRENIZACION': 'MODERNIZACION',

    'RIAS REGIONALES': 'VIAS REGIONALES',
    'VIA REGIONALES': 'VIAS REGIONALES',
    'VAIS REGIONALES': 'VIAS REGIONALES',
    'VISAS REGIONALES': 'VIAS REGIONALES',
    'VIAS REFGIONALES': 'VIAS REGIONALES',
    'VIAAS REGIONALES': 'VIAS REGIONALES',

    'G. REGIONAL': 'GRUPO REGIONAL',
    'GRUPO REGIOMNAL': 'GRUPO REGIONAL',
}

df[col_unidad] = df[col_unidad].replace(correcciones_unidad)

In [17]:
df.to_excel('interventoria_limpio.xlsx', index=False)